In [78]:
from google import genai
import chromadb
from dotenv import load_dotenv
import os

In [79]:
load_dotenv()

# Initialize the GenAI client
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
client = genai.Client(api_key=GOOGLE_API_KEY)

Manual chunking

In [80]:
from pypdf import PdfReader

def extract_text(file_path):
    # loading pdf
    reader = PdfReader(file_path)
    full_text = ""

    # retreive page texts
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            full_text += page_text + "\n"
    
    return full_text

def chunk_text(full_text,  chunk_size=500, chunk_overlap=50):
    chunks = []

    for i  in range(0, len(full_text) , chunk_size - chunk_overlap):
        chunk = full_text[i: i + chunk_size]
        chunks.append(chunk)
    return chunks

In [81]:
full_text = extract_text(file_path="karthi_profile_summary.pdf")
chunks = chunk_text(full_text=full_text)

Embedding

In [82]:
def embed_chunhs(chunks):
    embeddings = []
    for chunck in chunks:
        result = client.models.embed_content(
            model = "gemini-embedding-2",
            contents=chunck
        )
        embeddings.append(result.embeddings[0].values)
    return embeddings


In [ ]:
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name = "knowledge_base")

embeddings = embed_chunhs(chunks=chunks)

chunk_ids = [ f"chunk_{i}" for i in range(len(chunks))]

collection.add(
    embeddings=embeddings,
    ids=chunk_ids
)